# Generate environment

By joining different sources of data:
- Weather data for TMY
- Electricity price data
- Thermal load data

And save them as a unified dataset.

In [7]:
from pathlib import Path
import pandas as pd
import datetime
from loguru import logger

# %load_ext autoreload
# %autoreload 2

data_path: Path = Path("../../data/datasets")
output_path: Path = Path("../results")
weather_filename: str = "tmy_2005_tabernas.h5"
electricity_filename: str = "electricity_price_data_20211231_20241231.h5"
load_filename: str = "thermal_load_data_20220101_20241231.h5"
water_filename: str = "water_data_20220101_20241231.h5"

year_start: int = 2022
year_end: int = 2024


In [8]:
# Load datasets
df_weather = pd.read_hdf(data_path / weather_filename)
df_electricity = pd.read_hdf(data_path / electricity_filename)
df_load = pd.read_hdf(data_path / load_filename)
df_water = pd.read_hdf(data_path / water_filename)

# Create a pandas series with one-hour resolution
start_date = datetime.datetime(year_start, 1, 1, 0, 0, 0, tzinfo=datetime.timezone.utc)
end_date = datetime.datetime(year_end, 12, 31, 22, 0, 0, tzinfo=datetime.timezone.utc)
date_rng = pd.date_range(start=start_date, end=end_date, freq='h', tz='UTC')

display(df_weather.head())
display(df_electricity.head())
display(df_load.head())
display(df_water.head())


,G_Gh,G_Dh,G_Gk,G_Bn,Tamb_C,HR_pct,hs,Az,precip_mm,Td
time,,,,,,,,,,
2005-01-01 00:00:00+00:00,0,0,0,0,10.7,90,0.0,-144.0,6.7,9.1
2005-01-01 01:00:00+00:00,0,0,0,0,11.6,82,0.0,-163.4,0.5,8.7
2005-01-01 02:00:00+00:00,0,0,0,0,11.3,82,0.0,-124.4,0.5,8.4
2005-01-01 03:00:00+00:00,0,0,0,0,11.2,82,0.0,-105.4,0.3,8.3
2005-01-01 04:00:00+00:00,0,0,0,0,11.0,82,0.0,-93.6,0.2,8.1


,Ce_pvpc_eur_MWh,Ce_spot_market_price_eur_MWh,Ce_pvpc_eur_kWh,Ce_spot_market_price_eur_kWh
datetime,,,,
2021-12-31 23:00:00+00:00,204.51,145.86,0.20451,0.14586
2022-01-01 00:00:00+00:00,171.35,114.90,0.17135,0.11490
2022-01-01 01:00:00+00:00,172.70,113.87,0.17270,0.11387
2022-01-01 02:00:00+00:00,156.07,97.80,0.15607,0.09780
2022-01-01 03:00:00+00:00,159.08,97.80,0.15908,0.09780


,Q_kW,Tv_C
2022-01-01 00:00:00+00:00,200.0,35.0
2022-01-01 01:00:00+00:00,200.0,35.0
2022-01-01 02:00:00+00:00,200.0,35.0
2022-01-01 03:00:00+00:00,200.0,35.0
2022-01-01 04:00:00+00:00,200.0,35.0


,water_price_morocco_eur_m3,water_price_morocco_alternative_eur_m3,water_price_egypt_eur_m3,water_price_egypt_alternative_eur_m3,Vavail_m3,deltaV_m3_h,Vavail_l,deltaV_l_h
2022-01-01 00:00:00+00:00,0.029054,0.581078,0.060471,1.209412,19.083485,-0.916515,19083.485034,-916.514966
2022-01-01 01:00:00+00:00,0.029054,0.581078,0.060471,1.209412,19.020231,-0.979769,19020.230685,-979.769315
2022-01-01 02:00:00+00:00,0.029054,0.581078,0.060471,1.209412,19.002447,-0.997553,19002.447115,-997.552885
2022-01-01 03:00:00+00:00,0.029054,0.581078,0.060471,1.209412,18.965136,-1.034864,18965.135822,-1034.864178
2022-01-01 04:00:00+00:00,0.029054,0.581078,0.060471,1.209412,18.968172,-1.031828,18968.171943,-1031.828057


In [9]:
# Electricity price and thermal load data should be available for the selected period
# Raise an error if the data is not available
assert df_electricity.index[0] <= start_date and df_electricity.index[-1] >= end_date, f"Electricity price data is not available for the selected period: {df_electricity.index[0]} - {df_electricity.index[-1]}"
assert df_load.index[0] <= start_date and df_load.index[-1] >= end_date, f"Thermal load data is not available for the selected period: {df_load.index[0]} - {df_load.index[-1]}"

# Weather data needs to be adapted to the selected period (modify its year)
# Repeat data for each year
df_weather_ = df_weather.copy()
df_weather_ = pd.concat([df_weather_] * (year_end - year_start + 1))
df_weather_ = df_weather_.set_index(date_rng[:len(df_weather_)]).reindex(date_rng).ffill()

# Check if the weather data is available for the selected period
assert df_weather_.index[0] <= start_date and df_weather_.index[-1] >= end_date, f"Weather data is not available for the selected period: {df_weather.index[0]} - {df_weather.index[-1]}"

df_weather = df_weather_


In [10]:
# Join datasets
df = pd.concat([df_weather, df_electricity, df_load, df_water], axis=1, join="inner")

logger.info(f"Resulting dataframe has {len(df)} rows")

# Save to hdf5 and csv
fname = f"environment_data_{df.index[0].strftime('%Y%m%d')}_{df.index[-1].strftime('%Y%m%d')}"
df.sort_index(inplace=True)
df.to_hdf(data_path / f"{fname}.h5", key="data", mode="w", format="table")
df.to_csv(data_path / f"{fname}.csv")
    
logger.info(f"Saved {fname} to {data_path}")


2025-02-28 14:49:43.916 | INFO     | __main__:<module>:4 - Resulting dataframe has 26303 rows


2025-02-28 14:49:44.542 | INFO     | __main__:<module>:12 - Saved environment_data_20220101_20241231 to ../../data/datasets


In [11]:
df_env = pd.read_hdf(data_path / f"{fname}.h5")
df_env


,G_Gh,G_Dh,G_Gk,G_Bn,Tamb_C,HR_pct,hs,Az,precip_mm,Td,...,Q_kW,Tv_C,water_price_morocco_eur_m3,water_price_morocco_alternative_eur_m3,water_price_egypt_eur_m3,water_price_egypt_alternative_eur_m3,Vavail_m3,deltaV_m3_h,Vavail_l,deltaV_l_h
2022-01-01 00:00:00+00:00,0.0,0.0,0.0,0.0,10.7,90.0,0.0,-144.0,6.7,9.1,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.083485,-0.916515,19083.485034,-916.514966
2022-01-01 01:00:00+00:00,0.0,0.0,0.0,0.0,11.6,82.0,0.0,-163.4,0.5,8.7,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.020231,-0.979769,19020.230685,-979.769315
2022-01-01 02:00:00+00:00,0.0,0.0,0.0,0.0,11.3,82.0,0.0,-124.4,0.5,8.4,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.002447,-0.997553,19002.447115,-997.552885
2022-01-01 03:00:00+00:00,0.0,0.0,0.0,0.0,11.2,82.0,0.0,-105.4,0.3,8.3,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,18.965136,-1.034864,18965.135822,-1034.864178
2022-01-01 04:00:00+00:00,0.0,0.0,0.0,0.0,11.0,82.0,0.0,-93.6,0.2,8.1,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,18.968172,-1.031828,18968.171943,-1031.828057
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 18:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,5.7,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.047302,-0.952698,19047.301795,-952.698205
2024-12-31 19:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,5.7,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.090256,-0.909744,19090.255607,-909.744393
2024-12-31 20:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,5.7,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.048952,-0.951048,19048.952026,-951.047974
2024-12-31 21:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,5.7,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.098820,-0.901180,19098.819632,-901.180368


In [12]:
from plotly_resampler import FigureWidgetResampler
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

var_ids: list[str] = [
    ["Tamb_C", "Tv_C"], 
    "HR_pct", "precip_mm", "Q_kW", 
    ["Ce_pvpc_eur_MWh", "Ce_spot_market_price_eur_MWh"],
    ["water_price_morocco_eur_m3", "water_price_egypt_eur_m3", "water_price_morocco_alternative_eur_m3", "water_price_egypt_alternative_eur_m3"], 
    "Vavail_m3",]
var_units: list[str] = ["°C", "%", "mm", "kW<sub>th</sub>", "€/MW<sub>e</sub>", "€/m<sup>3</sup>","m<sup>3</sup>"]

fig = make_subplots(rows=len(var_ids), cols=1, shared_xaxes=True)
fig = FigureWidgetResampler(fig)

for i, (var_id, var_unit) in enumerate(zip(var_ids, var_units)):
    var_id = [var_id] if not isinstance(var_id, list) else var_id
    
    for v_id in var_id:
        fig.add_trace(
            go.Scattergl(name=v_id, showlegend=True), 
            hf_x=df_env.index, 
            hf_y=np.ascontiguousarray( df_env[v_id] ), 
            # max_n_samples=2_000,
            row=i + 1, col=1
        )
    fig.update_yaxes(title_text=var_unit, row=i + 1)

fig.update_layout(
    height=1400,
    width=800,
    title="<b>Environment variables</b>",
    title_x=0.1,
    legend_traceorder="normal",
    legend=dict(orientation="h", y=1.15, xanchor="left", x=0),
    margin=dict(l=20, r=20, t=340, b=20),
)

fig


FigureWidgetResampler({
    'data': [{'name': '<b style="color:sandybrown">[R]</b> Tamb_C <i style="color:#fc9944">~1D</i>',
              'showlegend': True,
              'type': 'scattergl',
              'uid': '3e900b86-c537-4cb3-a0c3-8d723fb285cd',
              'x': array([datetime.datetime(2022, 1, 1, 0, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2022, 1, 1, 13, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2022, 1, 3, 5, 0, tzinfo=datetime.timezone.utc), ...,
                          datetime.datetime(2024, 12, 29, 15, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2024, 12, 30, 23, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2024, 12, 31, 22, 0, tzinfo=datetime.timezone.utc)],
                         shape=(1000,), dtype=object),
              'xaxis': 'x',
              'y': array([10.7, 13.4,  5.6, ..., 15.3,  6. ,  6. ], shape=(1000,)),
    

In [13]:
from phd_visualizations import save_figure

start, end = fig.layout.xaxis.range
save_figure(fig, f"solhycool_env_viz_{start[:10].replace('-', '')}_{end[:10].replace('-', '')}", 
            figure_path=output_path, formats=["png", "svg"])


TypeError: cannot unpack non-iterable NoneType object